In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from datasets import Dataset

In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
dataset = Dataset.from_pandas(data)
dataset = dataset.train_test_split(test_size=0.3)
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 35000
    })
    test: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 15000
    })
})

In [4]:
data["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Convert string labels to int labels

In [5]:
label2id = {"positive": 1, "negative": 0}
id2label = {1: "positive", 0: "negative"}

dataset = dataset.map(lambda x: {"label": label2id[x["sentiment"]]})

Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment', 'label'],
        num_rows: 35000
    })
    test: Dataset({
        features: ['review', 'sentiment', 'label'],
        num_rows: 15000
    })
})

In [7]:
dataset["train"][0]

{'review': "Sergio Martino has impressed me recently with his Giallo classics 'The Strange Vice of Mrs Wardh' and the unforgettably titled, 'Your Vice is a Locked Room and Only I Have the Key' - but even so, I wasn't expecting too much from this film. The Case of the Scorpion's Tail doesn't get mentioned as much as the aforementioned titles when it comes to classic Giallo discussion - but I don't know why, because this is at least as good as those two! Dario Argento may be the 'king' of Giallo, but with the five films that he made - Sergio Martino surely isn't too far behind. In some ways, he even surpasses the master. All of Martino's films were released prior to the jewel in Argento's crown, the magnificent Profondo Rosso, so back in the early seventies - Martino was the king! The plot here follows the idea of murder for profit, and follows the insurance payout of a wealthy man. His wife inherits $1 million, and it isn't long before there's people out for her blood! When she turns up

Do the tokenization of the inputs.\
Therefore load tokenizer.

The 'TinyBERT_General_4L_312D' contains for 4 encoder layers and an embedding dimension of 312.

In [8]:
from transformers import AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_ckpt = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt, use_fast=True)

Define a tokenize function to use it for batch tokenization.

In [9]:
def tokenize(batch):
    return tokenizer(batch["review"], padding="max_length", truncation=True, max_length=512)

dataset = dataset.map(tokenize, batched=True, batch_size=None)

Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Dataset is expanded by input_ids and attention_mask and token_type_ids.

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 35000
    })
    test: Dataset({
        features: ['review', 'sentiment', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15000
    })
})

Create a function that computes the accuracy of the model.

In [11]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

When using 'AutoModelForSequenceClassification', the model gets a new trainable classification linear layer because usually the model is made for outputting token sequences.

In [12]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=len(label2id), id2label=id2label, label2id=label2id)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 312, padding_idx=0)
      (position_embeddings): Embedding(512, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-1

Setting all the training arguments

In [13]:
args = TrainingArguments(
    output_dir="train_dir",
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

Starting training

In [14]:
trainer.train()

  0%|          | 0/3282 [00:00<?, ?it/s]

{'loss': 0.4594, 'grad_norm': 19.84442901611328, 'learning_rate': 1.695307739183425e-05, 'epoch': 0.46}
{'loss': 0.3269, 'grad_norm': 15.797927856445312, 'learning_rate': 1.3906154783668494e-05, 'epoch': 0.91}


  0%|          | 0/469 [00:00<?, ?it/s]

{'eval_loss': 0.29794391989707947, 'eval_accuracy': 0.8764, 'eval_runtime': 108.7283, 'eval_samples_per_second': 137.959, 'eval_steps_per_second': 4.314, 'epoch': 1.0}
{'loss': 0.2912, 'grad_norm': 9.575115203857422, 'learning_rate': 1.0859232175502743e-05, 'epoch': 1.37}
{'loss': 0.2723, 'grad_norm': 15.550151824951172, 'learning_rate': 7.81230956733699e-06, 'epoch': 1.83}


  0%|          | 0/469 [00:00<?, ?it/s]

{'eval_loss': 0.2766828238964081, 'eval_accuracy': 0.8897333333333334, 'eval_runtime': 1148.5221, 'eval_samples_per_second': 13.06, 'eval_steps_per_second': 0.408, 'epoch': 2.0}
{'loss': 0.2487, 'grad_norm': 14.770716667175293, 'learning_rate': 4.765386959171238e-06, 'epoch': 2.29}
{'loss': 0.2369, 'grad_norm': 13.541827201843262, 'learning_rate': 1.7184643510054846e-06, 'epoch': 2.74}


  0%|          | 0/469 [00:00<?, ?it/s]

{'eval_loss': 0.2698921859264374, 'eval_accuracy': 0.8937333333333334, 'eval_runtime': 110.325, 'eval_samples_per_second': 135.962, 'eval_steps_per_second': 4.251, 'epoch': 3.0}
{'train_runtime': 7789.9824, 'train_samples_per_second': 13.479, 'train_steps_per_second': 0.421, 'train_loss': 0.30024077643859975, 'epoch': 3.0}


TrainOutput(global_step=3282, training_loss=0.30024077643859975, metrics={'train_runtime': 7789.9824, 'train_samples_per_second': 13.479, 'train_steps_per_second': 0.421, 'total_flos': 1505594603520000.0, 'train_loss': 0.30024077643859975, 'epoch': 3.0})

Save trained model

In [15]:
trainer.save_model("tinybert-sentiment-analysis")

Do some sample classification

In [20]:
from transformers import pipeline

data = ['this movie was horrible, the plot was really boring. acting was okay',
        'the movie is really sucked. there is not plot and acting was bad',
        'what a beautiful movie. great plot. acting was good. will see it again']

classifier = pipeline("sentiment-analysis", model="tinybert-sentiment-analysis", device=device)
classifier(data)

[{'label': 'negative', 'score': 0.9902814626693726},
 {'label': 'negative', 'score': 0.9900671243667603},
 {'label': 'positive', 'score': 0.990715742111206}]

Create a bucket and uploading the trained model to it.

In [ ]:
import boto3

s3 = boto3.client('s3', region_name='eu-north-1')

bucket_name = "mlops-mhofmann"

def create_bucket(bucket_name):
    """Create an S3 bucket in a specified region"""
    try:
        response = s3.list_buckets()
        if any(bucket['Name'] == bucket_name for bucket in response['Buckets']):
            print(f"Bucket {bucket_name} already exists.")
            return
        s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={
            'LocationConstraint': "eu-north-1"
        })
        print(f"Bucket {bucket_name} created successfully.")
    except Exception as e:
        print(f"Error creating bucket: {e}")

create_bucket(bucket_name)

In [ ]:
import os

def upload_file_to_s3(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket"""
    if object_name is None:
        object_name = os.path.basename(file_name)
    
    try:
        s3.upload_file(file_name, bucket, object_name)
        print(f"File {file_name} uploaded to {bucket}/{object_name}.")
    except Exception as e:
        print(f"Error uploading file: {e}")

def upload_directory_to_s3(directory, s3_prefix):
    """Upload a directory to an S3 bucket"""
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            relative_path = os.path.relpath(file_path, directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            # print(file_path, relative_path, s3_key)
            upload_file_to_s3(file_path, bucket_name, s3_key)

In [ ]:
upload_directory_to_s3("tinybert-sentiment-analysis", "ml-models/tinybert-sentiment-analysis")